# Retrieval-based Voice Conversion (RVC) WebUI - Google Colab 실행 노트

이 노트북은 Google Colab GPU 환경에서 RVC WebUI를 손쉽게 설치하고 실행하기 위한 예시입니다.

**[주의]** Colab 런타임 유형이 **GPU (T4, L4, A100 등)**로 설정되어 있는지 확인하세요.
(메뉴: `런타임` -> `런타임 유형 변경` -> `T4 GPU` 선택)

### Step 0. GPU 상태 확인

In [ ]:
!nvidia-smi

### Step 1. Google Drive 마운트 (선택 사항)
학습된 모델(`.pth`), 인덱스 파일(`.index`) 또는 음성 데이터를 구글 드라이브에 보관/불러오려면 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2. RVC 공식 저장소 클론 및 이동

In [ ]:
# 1. RVC 공식 저장소 클론
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
%cd Retrieval-based-Voice-Conversion-WebUI

### Step 3. 필수 패키지 및 의존성 라이브러리 설치

In [ ]:
# 시스템 패키지 및 파이썬 의존성 설치 (setuptools는 PyTorch 및 Gradio 호환을 위해 81 미만으로 고정)
!apt-get install -y ffmpeg libsndfile1
!pip install --upgrade pip "setuptools<81" wheel
# Google Colab 최신 GPU 환경(CUDA 12.x)에 맞춰 requirments_cu128_py312.txt 사용 권장
!pip install -r requirments_cu128_py312.txt --extra-index-url https://pypi.org/simple

### Step 4. 필수 사전 학습 모델 (Hubert / RMVPE / Pretrained) 다운로드
음성 추론 및 모델 학습에 필수적인 가중치 파일들을 Hugging Face에서 다운로드합니다.

In [ ]:
!pip install --upgrade huggingface_hub

# Hubert Base (음성 특징 추출용)
!hf download lj1995/VoiceConversionWebUI --revision main --include "hubert_base/*" --local-dir assets

# RMVPE (피치 추출용)
!hf download lj1995/VoiceConversionWebUI rmvpe.pt --revision main --local-dir assets/rmvpe

# Pretrained 모델 (학습용 베이스 모델 v1 / v2)
!hf download lj1995/VoiceConversionWebUI --revision main --include "pretrained/*" "pretrained_v2/*" --local-dir assets

# 무음 데이터 (학습 사전 작업용)
!hf download lj1995/VoiceConversionWebUI mute.zip --revision main --local-dir .model-downloads
!python -m zipfile -e .model-downloads/mute.zip logs

### Step 5. RVC WebUI 실행
`--colab` 옵션을 붙여 실행하면 Gradio에서 외부 접속이 가능한 `https://xxxx.gradio.live` 공개 URL 링크를 자동으로 생성해 줍니다.

In [ ]:
!python webui.py --colab